# Continue from your previous run — rebalanced test set, third model, symmetric Table 7

**Paste the cells below onto the end of your already-open, already-executed notebook
tab from the original ~3-hour training run — same runtime, same kernel.** Don't
re-run any of your original cells; your already-trained `yolo11m`/`yolo11n` weights
are still valid (only the `test` split changed) and this reuses them as-is, no
retraining needed for either of them.

**Assumption:** the Colab runtime from your original run is still connected, so
`DATA_DIR`, `YOLO_SEG_DIR`, `DIR_DATASET` still point at real directories on this
runtime's disk. If the runtime disconnected and you're on a fresh one instead,
re-run your original notebook's Setup/Configuration/repo-clone/data-conversion cells
first (fast — seconds, not the 3-hour training part) so those paths exist again, then
continue from here.

What this does, and why it's short:
1. Pulls the latest `lestojas/segmentation` branch, which now has the **test** split
   rebalanced to exact parity (89 crack / 89 crack-free images, was 89/13) — `train`
   and `valid` are untouched.
2. Rebuilds only the `test` portion of the YOLO-format datasets (cheap — symlinks +
   small label files, not a training step).
3. **Re-evaluates your existing main (`yolo11m`) and baseline (`yolo11n`) models**
   against the new, balanced test set — mandatory, since your original confusion
   matrix / IoU / direction numbers were computed against the old, imbalanced test
   set and are now stale. No retraining involved, just re-running `.val()`/`.predict()`.
4. Trains a **third, larger model** (`yolo11l-seg` / `yolo11l-cls`) — the only step
   that costs real GPU time — aimed at the weaker segmentation/direction results
   specifically, without touching detection.
5. Rebuilds all 8 research tables (Tables 0-7), including a reworked, **symmetric**
   Table 7 (two-sided pairwise tests between every pair of trained models, no model
   singled out as 'the' baseline) covering all three models.

## Step 1 — Imports & configuration

The `pip install` below is idempotent -- if these packages are already installed on
this runtime (i.e. you really are continuing the same live session), it's a fast
no-op; if this turned out to be a fresh runtime, it installs them fresh instead of
failing on `ModuleNotFoundError`.

In [ ]:
!pip install -q ultralytics scikit-learn seaborn

In [ ]:
import os, shutil, json, csv, random, itertools, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from sklearn.metrics import classification_report
from scipy import stats
from IPython.display import display

from ultralytics import YOLO

In [ ]:
# --- Configuration (safe to redefine even if your original run already set these) ---
REPO_URL = 'https://github.com/lestojas/segmentation.git'
BRANCH   = 'claude/lestojas-segmentation-direction-o9c5p8'

SEG_MODEL   = 'yolo11m-seg.pt'
CLS_MODEL   = 'yolo11m-cls.pt'
BASELINE_SEG_MODEL = 'yolo11n-seg.pt'
BASELINE_CLS_MODEL = 'yolo11n-cls.pt'
THIRD_SEG_MODEL = 'yolo11l-seg.pt'   # new: larger model, aimed at segmentation/direction
THIRD_CLS_MODEL = 'yolo11l-cls.pt'

IMG_SIZE    = 640
SEG_EPOCHS  = 100
CLS_EPOCHS  = 50
BATCH       = -1
CONF_THRES  = 0.25
FORCE_RETRAIN = False  # set True to retrain the third model even if it's already trained
SAVE_TO_DRIVE = False

assert 'DATA_DIR' in dir() and os.path.isdir(DATA_DIR), (
    "DATA_DIR isn't set or doesn't exist -- this runtime looks like it doesn't have your "
    "original session's state. Re-run your original notebook's Setup/Configuration/"
    "repo-clone/data-conversion cells first (fast), then re-run this cell.")
print('Continuing on existing DATA_DIR:', DATA_DIR)

## Step 2 — Pull the rebalanced test split

`DATA_DIR` is a disposable clone (not your working repo), so a hard reset is safe --
it just brings it fully up to date with the branch, including the new parity-balanced
`test` split.

In [ ]:
!git -C {DATA_DIR} fetch origin {BRANCH}
!git -C {DATA_DIR} reset --hard origin/{BRANCH}
print('Repo refreshed to latest', BRANCH, '-- test split is now rebalanced to parity.')

## Step 3 — Rebuild only the `test` portion of the YOLO-format datasets

`train`/`valid` didn't change, so they're left alone. `test` is deleted and rebuilt
from scratch (cheap: symlinks + small label files, ~178 images, not a GPU step).

In [ ]:
def convert_split_to_yolo_seg(split):
    coco = json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json'))
    anns_by_image = defaultdict(list)
    for a in coco['annotations']:
        anns_by_image[a['image_id']].append(a)

    img_out = YOLO_SEG_DIR / split / 'images'
    lbl_out = YOLO_SEG_DIR / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for im in coco['images']:
        w, h = im['width'], im['height']
        src = Path(DATA_DIR) / split / im['file_name']
        dst = img_out / im['file_name']
        if not dst.exists():
            os.symlink(src, dst)

        lines = []
        for ann in anns_by_image.get(im['id'], []):
            for seg in ann.get('segmentation', []):
                if len(seg) < 6:
                    continue
                coords = []
                for i in range(0, len(seg), 2):
                    x = min(max(seg[i] / w, 0.0), 1.0)
                    y = min(max(seg[i + 1] / h, 0.0), 1.0)
                    coords.append(f'{x:.6f} {y:.6f}')
                lines.append('0 ' + ' '.join(coords))

        (lbl_out / (Path(im['file_name']).stem + '.txt')).write_text('\n'.join(lines))

    return len(coco['images']), sum(len(v) for v in anns_by_image.values())

YOLO_SEG_DIR = Path('/content/yolo_seg_dataset')
DIR_DATASET = Path('/content/direction_dataset')
SPLITS = ['train', 'valid', 'test']

yaml_path = YOLO_SEG_DIR / 'data.yaml'
if not yaml_path.exists():
    raise FileNotFoundError(
        f'{yaml_path} not found -- train/valid conversion is missing too, meaning this is '
        "a fresh runtime, not a continued one. Re-run your original notebook's Sections "
        '0-2 first, then re-run this cell.')

for sub in [YOLO_SEG_DIR / 'test', DIR_DATASET / 'test']:
    if sub.exists():
        shutil.rmtree(sub)

n_img, n_ann = convert_split_to_yolo_seg('test')
print(f'test: {n_img} images, {n_ann} annotations converted (YOLO-seg format)')

with open(f'{DATA_DIR}/test/_direction_labels.csv') as f:
    dir_rows = list(csv.DictReader(f))
for r in dir_rows:
    cls_dir = DIR_DATASET / 'test' / r['direction']
    cls_dir.mkdir(parents=True, exist_ok=True)
    src = Path(DATA_DIR) / 'test' / r['file_name']
    dst = cls_dir / r['file_name']
    if not dst.exists():
        os.symlink(src, dst)
print(f"test: {len(dir_rows)} direction-labeled images rebuilt under {DIR_DATASET / 'test'}")

# Refresh direction_df / all_coco across all splits (cheap) so Tables 0-1 reflect the
# new test-split counts.
direction_rows = []
for split in SPLITS:
    with open(f'{DATA_DIR}/{split}/_direction_labels.csv') as f:
        for r in csv.DictReader(f):
            r['split'] = split
            direction_rows.append(r)
direction_df = pd.DataFrame(direction_rows)
direction_df['num_annotations'] = direction_df['num_annotations'].astype(int)
direction_df['angle_deg'] = pd.to_numeric(direction_df['angle_deg'], errors='coerce')
direction_df['elongation_ratio'] = pd.to_numeric(direction_df['elongation_ratio'], errors='coerce')

all_coco = {split: json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json')) for split in SPLITS}
print(direction_df.groupby(['split', 'direction']).size().unstack(fill_value=0))

## Step 4 — Re-evaluate the existing main + baseline models on the new test set

No retraining here -- `runs/crack_seg/weights/best.pt` and
`runs/crack_seg_baseline/weights/best.pt` are reloaded as-is and just re-scored
against the refreshed, balanced `test` split.

In [ ]:
def require_weights(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f'{p} not found. If this model\'s weights are no longer on this runtime\'s '
            "disk, see the weight-recovery guidance in Section 0 of the main notebook "
            "(colab/train_crack_direction_model.ipynb) to re-upload them, or check "
            "Google Drive if you used SAVE_TO_DRIVE in the original run.")
    return str(p)

def image_level_eval(model, conf_thres=CONF_THRES):
    preds = model.predict(source=[str(p) for p in test_files], conf=conf_thres,
                           imgsz=IMG_SIZE, verbose=False)
    y_t, y_p = [], []
    for p in preds:
        fname = Path(p.path).name
        y_t.append(filename_to_gt.get(fname, False))
        y_p.append(len(p.boxes) > 0)
    return preds, np.array(y_t), np.array(y_p)

def confusion_metrics(y_t, y_p):
    tp = int(np.sum(y_t & y_p)); fn = int(np.sum(y_t & ~y_p))
    fp = int(np.sum(~y_t & y_p)); tn = int(np.sum(~y_t & ~y_p))
    acc = (tp + tn) / (tp + tn + fp + fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    rec = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else np.nan
    return tp, fn, fp, tn, acc, prec, rec, spec, f1

best_seg = YOLO(require_weights('runs/crack_seg/weights/best.pt'))
best_seg_baseline = YOLO(require_weights('runs/crack_seg_baseline/weights/best.pt'))

seg_val = best_seg.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)
box_map50, box_map50_95 = seg_val.box.map50, seg_val.box.map
box_precision, box_recall = seg_val.box.mp, seg_val.box.mr

seg_val_baseline = best_seg_baseline.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)
baseline_box_map50, baseline_box_map50_95 = seg_val_baseline.box.map50, seg_val_baseline.box.map
baseline_box_precision, baseline_box_recall = seg_val_baseline.box.mp, seg_val_baseline.box.mr

test_images_dir = YOLO_SEG_DIR / 'test' / 'images'
test_files = sorted(test_images_dir.glob('*.jpg'))

coco_test = json.load(open(f'{DATA_DIR}/test/_annotations.coco.json'))
gt_has_crack = defaultdict(bool)
for a in coco_test['annotations']:
    gt_has_crack[a['image_id']] = True
filename_to_gt = {im['file_name']: gt_has_crack.get(im['id'], False) for im in coco_test['images']}

main_test_preds, y_true, y_pred = image_level_eval(best_seg)
(tp, fn, fp, tn, detection_accuracy, detection_precision, detection_recall,
 detection_specificity, detection_f1) = confusion_metrics(y_true, y_pred)

baseline_test_preds, _, y_pred_baseline = image_level_eval(best_seg_baseline)
(tp_b, fn_b, fp_b, tn_b, baseline_detection_accuracy, baseline_detection_precision,
 baseline_detection_recall, baseline_detection_specificity, baseline_detection_f1) = (
    confusion_metrics(y_true, y_pred_baseline))

print(f'Test images: {len(y_true)}  (crack: {int(y_true.sum())}, crack-free: {int((~y_true).sum())})')
print(f'Main     ({SEG_MODEL}) -- Accuracy: {detection_accuracy:.4f}  '
      f'TP:{tp} FN:{fn} FP:{fp} TN:{tn}')
print(f'Baseline ({BASELINE_SEG_MODEL}) -- Accuracy: {baseline_detection_accuracy:.4f}  '
      f'TP:{tp_b} FN:{fn_b} FP:{fp_b} TN:{tn_b}')

In [ ]:
def polygon_mask(segmentation, w, h):
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    for seg in segmentation:
        pts = list(zip(seg[0::2], seg[1::2]))
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(mask, dtype=bool)

def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union > 0 else 0.0

id_by_name = {im['file_name']: im for im in coco_test['images']}
anns_by_image_test = defaultdict(list)
for a in coco_test['annotations']:
    anns_by_image_test[a['image_id']].append(a)

def best_matched_ious(preds):
    ious_list = []
    for p in preds:
        fname = Path(p.path).name
        im_meta = id_by_name.get(fname)
        if im_meta is None:
            continue
        w, h = im_meta['width'], im_meta['height']
        gt_masks = [polygon_mask(a['segmentation'], w, h) for a in anns_by_image_test.get(im_meta['id'], [])]
        if not gt_masks:
            continue
        if p.masks is None:
            ious_list.extend([0.0] * len(gt_masks))
            continue
        pred_masks = [polygon_mask([poly.reshape(-1).tolist()], w, h) for poly in p.masks.xy]
        used = set()
        for gt in gt_masks:
            best_iou, best_j = 0.0, None
            for j, pm in enumerate(pred_masks):
                if j in used:
                    continue
                iou = mask_iou(gt, pm)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_j is not None:
                used.add(best_j)
            ious_list.append(best_iou)
    return np.array(ious_list)

seg_map50, seg_map50_95 = seg_val.seg.map50, seg_val.seg.map
seg_precision, seg_recall = seg_val.seg.mp, seg_val.seg.mr
baseline_seg_map50, baseline_seg_map50_95 = seg_val_baseline.seg.map50, seg_val_baseline.seg.map
baseline_seg_precision, baseline_seg_recall = seg_val_baseline.seg.mp, seg_val_baseline.seg.mr

ious = best_matched_ious(main_test_preds)
ious_baseline = best_matched_ious(baseline_test_preds)
print(f'Main mean mask IoU: {ious.mean():.4f}   Baseline mean mask IoU: {ious_baseline.mean():.4f}')

In [ ]:
best_cls = YOLO(require_weights('runs/crack_direction_cls/weights/best.pt'))
best_cls_baseline = YOLO(require_weights('runs/crack_direction_cls_baseline/weights/best.pt'))

test_dir_dataset = DIR_DATASET / 'test'
y_true_dir, files = [], []
for cls_name in sorted(os.listdir(test_dir_dataset)):
    for f in (test_dir_dataset / cls_name).glob('*.jpg'):
        files.append(f)
        y_true_dir.append(cls_name)

def direction_eval(model):
    class_names = model.names
    preds = model.predict(source=[str(f) for f in files], imgsz=224, verbose=False)
    return [class_names[int(p.probs.top1)] for p in preds]

y_pred_dir = direction_eval(best_cls)
y_pred_dir_baseline = direction_eval(best_cls_baseline)

y_true_dir_arr = np.array(y_true_dir)
y_pred_dir_arr = np.array(y_pred_dir)
dir_acc = (y_true_dir_arr == y_pred_dir_arr).mean()
baseline_dir_accuracy = (y_true_dir_arr == np.array(y_pred_dir_baseline)).mean()

report_dict = classification_report(y_true_dir, y_pred_dir, output_dict=True, digits=4)
macro = report_dict['macro avg']
baseline_report = classification_report(y_true_dir, y_pred_dir_baseline, output_dict=True, digits=4)
baseline_macro = baseline_report['macro avg']

print(f'Main top-1 accuracy: {dir_acc:.4f}   Baseline top-1 accuracy: {baseline_dir_accuracy:.4f}')

## Step 5 — Train (if needed) and evaluate the third, larger model

The only step here that costs real GPU time. Skips straight to evaluation if
`runs/crack_seg_third/weights/best.pt` (or the cls equivalent) already exists from an
earlier run of this notebook (set `FORCE_RETRAIN = True` above to always retrain).

In [ ]:
third_seg_model_path = Path('runs/crack_seg_third/weights/best.pt')
if FORCE_RETRAIN or not third_seg_model_path.exists():
    third_seg_model = YOLO(THIRD_SEG_MODEL)
    third_seg_model.train(
        data=str(yaml_path),
        epochs=SEG_EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        patience=20,
        cache='ram',
        cos_lr=True,
        amp=True,
        project='runs',
        name='crack_seg_third',
        seed=0,
    )
else:
    print(f'{third_seg_model_path} already exists -- skipping training (FORCE_RETRAIN=False).')

third_cls_model_path = Path('runs/crack_direction_cls_third/weights/best.pt')
if FORCE_RETRAIN or not third_cls_model_path.exists():
    third_cls_model = YOLO(THIRD_CLS_MODEL)
    third_cls_model.train(
        data=str(DIR_DATASET),
        epochs=CLS_EPOCHS,
        imgsz=224,
        batch=BATCH if BATCH != -1 else 64,
        patience=20,
        cache='ram',
        cos_lr=True,
        amp=True,
        project='runs',
        name='crack_direction_cls_third',
        seed=0,
    )
else:
    print(f'{third_cls_model_path} already exists -- skipping training (FORCE_RETRAIN=False).')

In [ ]:
best_seg_third = YOLO(require_weights('runs/crack_seg_third/weights/best.pt'))
seg_val_third = best_seg_third.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)
third_box_map50, third_box_map50_95 = seg_val_third.box.map50, seg_val_third.box.map
third_box_precision, third_box_recall = seg_val_third.box.mp, seg_val_third.box.mr
third_seg_map50, third_seg_map50_95 = seg_val_third.seg.map50, seg_val_third.seg.map
third_seg_precision, third_seg_recall = seg_val_third.seg.mp, seg_val_third.seg.mr

third_test_preds, _, y_pred_third = image_level_eval(best_seg_third)
(tp_t, fn_t, fp_t, tn_t, detection_accuracy_third, detection_precision_third,
 detection_recall_third, detection_specificity_third, detection_f1_third) = (
    confusion_metrics(y_true, y_pred_third))
ious_third = best_matched_ious(third_test_preds)

best_cls_third = YOLO(require_weights('runs/crack_direction_cls_third/weights/best.pt'))
y_pred_dir_third = direction_eval(best_cls_third)
third_dir_accuracy = (y_true_dir_arr == np.array(y_pred_dir_third)).mean()
third_report = classification_report(y_true_dir, y_pred_dir_third, output_dict=True, digits=4)
third_macro = third_report['macro avg']

print(f'Third ({THIRD_SEG_MODEL}) -- Detection accuracy: {detection_accuracy_third:.4f}  '
      f'Mean mask IoU: {ious_third.mean():.4f}')
print(f'Third ({THIRD_CLS_MODEL}) -- Direction top-1 accuracy: {third_dir_accuracy:.4f}')

## Step 6 — Export research-paper-ready tables (Tables 0-7)

Identical structure and column/row semantics to Section 10 of the main notebook
(`colab/train_crack_direction_model.ipynb`) -- 3-column descriptive tables and a
symmetric, two-sided pairwise Table 7 across all three models -- so the output is
consistent whether you got here via a fresh full run or this continuation.

In [ ]:
PAPER_TABLES_DIR = Path('/content/paper_tables')
PAPER_TABLES_DIR.mkdir(exist_ok=True)
exported_tables = {}

def register_table(name, df):
    exported_tables[name] = df
    return df

MAIN_SEG_LABEL = Path(SEG_MODEL).stem
BASELINE_SEG_LABEL = Path(BASELINE_SEG_MODEL).stem
THIRD_SEG_LABEL = Path(THIRD_SEG_MODEL).stem
MAIN_CLS_LABEL = Path(CLS_MODEL).stem
BASELINE_CLS_LABEL = Path(BASELINE_CLS_MODEL).stem
THIRD_CLS_LABEL = Path(THIRD_CLS_MODEL).stem

In [ ]:
area_records = []
no_crack_counts = {split: 0 for split in SPLITS}
for split, coco in all_coco.items():
    imgs_by_id = {im['id']: im for im in coco['images']}
    for im in coco['images']:
        if im.get('direction') == 'None':
            no_crack_counts[split] += 1
    for a in coco['annotations']:
        im = imgs_by_id[a['image_id']]
        area_records.append({
            'split': split,
            'direction': im.get('direction', 'Unknown'),
            'area_px': a['area'],
            'area_pct': 100 * a['area'] / (im['width'] * im['height']),
        })
area_df = pd.DataFrame(area_records)

table0 = pd.DataFrame([
    {'split': split, 'n_images': len(all_coco[split]['images']),
     'n_cracked_images': len(all_coco[split]['images']) - no_crack_counts[split],
     'n_no_crack_images': no_crack_counts[split]}
    for split in SPLITS
])
totals = table0[['n_images', 'n_cracked_images', 'n_no_crack_images']].sum().to_dict()
table0 = pd.concat([table0, pd.DataFrame([{'split': 'All', **totals}])], ignore_index=True)
register_table('Table0_dataset_split_summary', table0)
display(table0)

In [ ]:
image_counts = direction_df.groupby(['split', 'direction']).size().rename('n_images').reset_index()
ann_counts = (direction_df.groupby(['split', 'direction'])['num_annotations'].sum()
              .rename('n_annotations').reset_index())
area_stats = (area_df.groupby(['split', 'direction'])['area_pct']
              .agg(['count', 'mean', 'std', 'median', 'min', 'max']).reset_index())
area_stats.columns = ['split', 'direction', 'n_instances', 'mean_area_pct', 'sd_area_pct',
                       'median_area_pct', 'min_area_pct', 'max_area_pct']

table1_cracked = (image_counts.merge(ann_counts, on=['split', 'direction'])
                  .merge(area_stats, on=['split', 'direction']))

no_crack_rows = pd.DataFrame([
    {'split': split, 'direction': 'No Crack', 'n_images': no_crack_counts[split], 'n_annotations': 0,
     'n_instances': 0, 'mean_area_pct': np.nan, 'sd_area_pct': np.nan, 'median_area_pct': np.nan,
     'min_area_pct': np.nan, 'max_area_pct': np.nan}
    for split in SPLITS
])

table1_all_splits = (pd.concat([table1_cracked, no_crack_rows], ignore_index=True)
                      .sort_values(['split', 'direction']).reset_index(drop=True).round(3))
table1 = (table1_all_splits[table1_all_splits['split'] == 'test']
          .drop(columns='split').reset_index(drop=True))
register_table('Table1_test_split_composition', table1)
display(table1)

In [ ]:
cm_rows = [
    {'Model': MAIN_SEG_LABEL, 'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn, 'N': len(y_true)},
    {'Model': BASELINE_SEG_LABEL, 'TP': tp_b, 'FN': fn_b, 'FP': fp_b, 'TN': tn_b, 'N': len(y_true)},
    {'Model': THIRD_SEG_LABEL, 'TP': tp_t, 'FN': fn_t, 'FP': fp_t, 'TN': tn_t, 'N': len(y_true)},
]
table2_cm = pd.DataFrame(cm_rows)
register_table('Table2_detection_confusion_matrix', table2_cm)
display(table2_cm)

In [ ]:
table3_det_img = pd.DataFrame([
    {'Metric': 'Accuracy', MAIN_SEG_LABEL: detection_accuracy, BASELINE_SEG_LABEL: baseline_detection_accuracy, THIRD_SEG_LABEL: detection_accuracy_third},
    {'Metric': 'Precision', MAIN_SEG_LABEL: detection_precision, BASELINE_SEG_LABEL: baseline_detection_precision, THIRD_SEG_LABEL: detection_precision_third},
    {'Metric': 'Recall', MAIN_SEG_LABEL: detection_recall, BASELINE_SEG_LABEL: baseline_detection_recall, THIRD_SEG_LABEL: detection_recall_third},
    {'Metric': 'Specificity', MAIN_SEG_LABEL: detection_specificity, BASELINE_SEG_LABEL: baseline_detection_specificity, THIRD_SEG_LABEL: detection_specificity_third},
    {'Metric': 'F1-score', MAIN_SEG_LABEL: detection_f1, BASELINE_SEG_LABEL: baseline_detection_f1, THIRD_SEG_LABEL: detection_f1_third},
]).round(4)
register_table('Table3_detection_image_level_metrics', table3_det_img)
display(table3_det_img)

In [ ]:
table4_det_box = pd.DataFrame([
    {'Metric': 'Box precision', MAIN_SEG_LABEL: box_precision, BASELINE_SEG_LABEL: baseline_box_precision, THIRD_SEG_LABEL: third_box_precision},
    {'Metric': 'Box recall', MAIN_SEG_LABEL: box_recall, BASELINE_SEG_LABEL: baseline_box_recall, THIRD_SEG_LABEL: third_box_recall},
    {'Metric': 'Box mAP50', MAIN_SEG_LABEL: box_map50, BASELINE_SEG_LABEL: baseline_box_map50, THIRD_SEG_LABEL: third_box_map50},
    {'Metric': 'Box mAP50-95', MAIN_SEG_LABEL: box_map50_95, BASELINE_SEG_LABEL: baseline_box_map50_95, THIRD_SEG_LABEL: third_box_map50_95},
]).round(4)
register_table('Table4_detection_box_level_metrics', table4_det_box)
display(table4_det_box)

In [ ]:
table5_seg = pd.DataFrame([
    {'Metric': 'Mean mask IoU', MAIN_SEG_LABEL: ious.mean(), BASELINE_SEG_LABEL: ious_baseline.mean(), THIRD_SEG_LABEL: ious_third.mean()},
    {'Metric': 'Mask precision', MAIN_SEG_LABEL: seg_precision, BASELINE_SEG_LABEL: baseline_seg_precision, THIRD_SEG_LABEL: third_seg_precision},
    {'Metric': 'Mask recall', MAIN_SEG_LABEL: seg_recall, BASELINE_SEG_LABEL: baseline_seg_recall, THIRD_SEG_LABEL: third_seg_recall},
    {'Metric': 'Mask mAP50', MAIN_SEG_LABEL: seg_map50, BASELINE_SEG_LABEL: baseline_seg_map50, THIRD_SEG_LABEL: third_seg_map50},
    {'Metric': 'Mask mAP50-95', MAIN_SEG_LABEL: seg_map50_95, BASELINE_SEG_LABEL: baseline_seg_map50_95, THIRD_SEG_LABEL: third_seg_map50_95},
]).round(4)
register_table('Table5_segmentation_performance', table5_seg)
display(table5_seg)

In [ ]:
table6_dir = pd.DataFrame([
    {'Metric': 'Top-1 accuracy', MAIN_CLS_LABEL: dir_acc, BASELINE_CLS_LABEL: baseline_dir_accuracy, THIRD_CLS_LABEL: third_dir_accuracy},
    {'Metric': 'Macro precision', MAIN_CLS_LABEL: macro['precision'], BASELINE_CLS_LABEL: baseline_macro['precision'], THIRD_CLS_LABEL: third_macro['precision']},
    {'Metric': 'Macro recall', MAIN_CLS_LABEL: macro['recall'], BASELINE_CLS_LABEL: baseline_macro['recall'], THIRD_CLS_LABEL: third_macro['recall']},
    {'Metric': 'Macro F1', MAIN_CLS_LABEL: macro['f1-score'], BASELINE_CLS_LABEL: baseline_macro['f1-score'], THIRD_CLS_LABEL: third_macro['f1-score']},
]).round(4)
register_table('Table6_direction_performance', table6_dir)
display(table6_dir)

### Table 7 — symmetric pairwise comparisons (no baseline framing)

Every pair of the three models is compared with a two-sided test (McNemar's for the
per-image detection/direction outcomes, Wilcoxon signed-rank for the paired IoUs) --
9 rows total. `Higher-performing model` is descriptive only, computed after the test.

In [ ]:
def mcnemar_test(correct_a, correct_b):
    correct_a, correct_b = np.asarray(correct_a), np.asarray(correct_b)
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    n = b + c
    if n == 0:
        return np.nan, 1.0, b, c
    if n < 25:
        p = stats.binomtest(b, n, 0.5, alternative='two-sided').pvalue
        stat = np.nan
    else:
        stat = (abs(b - c) - 1) ** 2 / n
        p = 1 - stats.chi2.cdf(stat, df=1)
    return stat, p, b, c

def higher_performing(label_a, score_a, label_b, score_b):
    if score_a == score_b:
        return 'Tie'
    return label_a if score_a > score_b else label_b

seg_models = [
    (MAIN_SEG_LABEL, y_true == y_pred, detection_accuracy, ious, ious.mean()),
    (BASELINE_SEG_LABEL, y_true == y_pred_baseline, baseline_detection_accuracy, ious_baseline, ious_baseline.mean()),
    (THIRD_SEG_LABEL, y_true == y_pred_third, detection_accuracy_third, ious_third, ious_third.mean()),
]
cls_models = [
    (MAIN_CLS_LABEL, y_true_dir_arr == y_pred_dir_arr, dir_acc),
    (BASELINE_CLS_LABEL, y_true_dir_arr == np.array(y_pred_dir_baseline), baseline_dir_accuracy),
    (THIRD_CLS_LABEL, y_true_dir_arr == np.array(y_pred_dir_third), third_dir_accuracy),
]

rows = []

for (label_a, correct_a, acc_a, _, _), (label_b, correct_b, acc_b, _, _) in itertools.combinations(seg_models, 2):
    stat, p, b, c = mcnemar_test(correct_a, correct_b)
    rows.append({'Task': 'Crack detection', 'Model A': label_a, 'Model B': label_b,
                 'Model A score': round(acc_a, 4), 'Model B score': round(acc_b, 4),
                 'Test': "McNemar's test", 'Statistic': stat, 'p_value': p,
                 'Significant (p<0.05)': p < 0.05,
                 'Higher-performing model': higher_performing(label_a, acc_a, label_b, acc_b)})

for (label_a, _, _, ious_a, mean_a), (label_b, _, _, ious_b, mean_b) in itertools.combinations(seg_models, 2):
    iou_diffs = ious_a - ious_b
    if np.any(iou_diffs != 0):
        seg_stat, seg_p = stats.wilcoxon(iou_diffs, alternative='two-sided')
    else:
        seg_stat, seg_p = np.nan, np.nan
    rows.append({'Task': 'Crack segmentation', 'Model A': label_a, 'Model B': label_b,
                 'Model A score': round(mean_a, 4), 'Model B score': round(mean_b, 4),
                 'Test': 'Wilcoxon signed-rank', 'Statistic': seg_stat, 'p_value': seg_p,
                 'Significant (p<0.05)': seg_p < 0.05,
                 'Higher-performing model': higher_performing(label_a, mean_a, label_b, mean_b)})

for (label_a, correct_a, acc_a), (label_b, correct_b, acc_b) in itertools.combinations(cls_models, 2):
    stat, p, b, c = mcnemar_test(correct_a, correct_b)
    rows.append({'Task': 'Direction classification', 'Model A': label_a, 'Model B': label_b,
                 'Model A score': round(acc_a, 4), 'Model B score': round(acc_b, 4),
                 'Test': "McNemar's test", 'Statistic': stat, 'p_value': p,
                 'Significant (p<0.05)': p < 0.05,
                 'Higher-performing model': higher_performing(label_a, acc_a, label_b, acc_b)})

table7 = pd.DataFrame(rows)
table7[['Statistic', 'p_value']] = table7[['Statistic', 'p_value']].round(4)
register_table('Table7_pairwise_model_comparisons', table7)
display(table7)

In [ ]:
for name, df in exported_tables.items():
    df.to_csv(PAPER_TABLES_DIR / f'{name}.csv', index=False)
    try:
        (PAPER_TABLES_DIR / f'{name}.tex').write_text(
            df.to_latex(index=False, float_format='%.4f', na_rep='--'))
    except Exception as e:
        print(f'Could not export {name} to LaTeX ({e}); CSV was still written.')

print(f'Exported {len(exported_tables)} tables to {PAPER_TABLES_DIR}:')
for f in sorted(PAPER_TABLES_DIR.iterdir()):
    print(' ', f.name)

zip_path = '/content/paper_tables.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in PAPER_TABLES_DIR.iterdir():
        zf.write(f, arcname=f.name)

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except Exception as e:
    print(f'Automatic browser download not available in this environment ({e}). '
          f'The files are still saved at {PAPER_TABLES_DIR} and {zip_path}.')

## Step 7 — Download all six models' best.pt weights

Bundles the best checkpoint for **all three seg models and all three cls models --
including the main + baseline ones from your original run**, not just the new third
model -- into one zip and triggers a browser download, so you can get every trained
model out of this runtime in one step.

In [ ]:
WEIGHTS_DIR = Path('/content/best_weights')
WEIGHTS_DIR.mkdir(exist_ok=True)

weight_sources = {
    f'crack_seg__{MAIN_SEG_LABEL}': 'runs/crack_seg/weights/best.pt',
    f'crack_seg_baseline__{BASELINE_SEG_LABEL}': 'runs/crack_seg_baseline/weights/best.pt',
    f'crack_seg_third__{THIRD_SEG_LABEL}': 'runs/crack_seg_third/weights/best.pt',
    f'crack_direction_cls__{MAIN_CLS_LABEL}': 'runs/crack_direction_cls/weights/best.pt',
    f'crack_direction_cls_baseline__{BASELINE_CLS_LABEL}': 'runs/crack_direction_cls_baseline/weights/best.pt',
    f'crack_direction_cls_third__{THIRD_CLS_LABEL}': 'runs/crack_direction_cls_third/weights/best.pt',
}

found, missing = [], []
for label, src in weight_sources.items():
    src_path = Path(src)
    if src_path.exists():
        dst = WEIGHTS_DIR / f'{label}_best.pt'
        shutil.copy(src_path, dst)
        found.append(dst.name)
    else:
        missing.append(src)

print(f'Bundled {len(found)} weight file(s):')
for name in found:
    print(' ', name)
if missing:
    print(f'Missing (not found on disk, skipped): {missing}')

weights_zip_path = '/content/best_weights.zip'
with zipfile.ZipFile(weights_zip_path, 'w') as zf:
    for f in WEIGHTS_DIR.iterdir():
        zf.write(f, arcname=f.name)

try:
    from google.colab import files as colab_files
    colab_files.download(weights_zip_path)
except Exception as e:
    print(f'Automatic browser download not available in this environment ({e}). '
          f'The weights are still saved at {WEIGHTS_DIR} and {weights_zip_path}.')

## (Optional) Also save weights & tables to Google Drive

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = '/content/drive/MyDrive/crack_models'
    os.makedirs(out_dir, exist_ok=True)
    for label, src in weight_sources.items():
        src_path = Path(src)
        if src_path.exists():
            shutil.copy(src_path, f'{out_dir}/{label}_best.pt')
    tables_out_dir = f'{out_dir}/paper_tables'
    if os.path.isdir(tables_out_dir):
        shutil.rmtree(tables_out_dir)
    shutil.copytree(PAPER_TABLES_DIR, tables_out_dir)
    print(f'Saved weights and paper tables to {out_dir}')
else:
    print('SAVE_TO_DRIVE is False -- skipping. Weights remain at runs/<run_name>/weights/best.pt '
          f'for each trained model, and exported tables remain at {PAPER_TABLES_DIR} / '
          '/content/paper_tables.zip for this session (Step 7 above already bundled and '
          'downloaded the weights to your browser).')